In [10]:
from dotenv import load_dotenv
load_dotenv()

True

# 工具定义

In [11]:
import time
import requests
def request_with_retry(url, params=None, max_retry=3):
    retry = 0
    success = False
    while not success and retry <= max_retry:
        response = requests.get(url, params)
        data = response.json()
        status = data['status']
        info = data['info']
        if status == '1':
            break
        else:
            retry += 1
            print(f"request failed, status={status}, info={info}, retry={retry}, max_retry={max_retry}, url={url}, param={params}.")
            time.sleep(retry)
    return data

In [12]:
# 工具1 : get_famous_spots
import requests
import os
import json


def get_famous_spots_for_page(city, page, rating):
    """获取城市热门景点"""
    amap_api_key = os.environ["AMAP_API_KEY"]
    
    # POI 分类编码：风景名胜
    # 参考：https://lbs.amap.com/api/webservice/download
    types = "风景名胜"
    params = {
        "key": amap_api_key,
        "types": "风景名胜",
        "city": city,
        "citylimit": True, 
        "page": page,
        "offset": 20,
        "extension": "all"
    }
    url = f"https://restapi.amap.com/v3/place/text"

    data = request_with_retry(url, params) 

    if data == None:
        return "未查到热门景点"
    else:
        pois = data['pois']

        # 取评分超过4.5的
        famous_pois = [{
            'name': poi['name'],
            'address': poi['address'],
            'cityname': poi['cityname'],
            'tel': poi['tel'],
            'location': poi['location'],
            'rating': poi['biz_ext']['rating'],
            'cost': poi['biz_ext']['cost']
        } for poi in pois if poi['biz_ext'].get('rating') and float(poi['biz_ext']['rating']) >= rating]

        return famous_pois


def get_famous_spots(city, rating=4.5): 
    all_pois = []
    for page in range(0, 1): 
        all_pois.extend(get_famous_spots_for_page(city, page, rating))
    return json.dumps(all_pois, ensure_ascii=False, indent=4)


In [13]:
# 工具2: get_future_weather
def get_adcode(city): 
    """获取行政区域编码adcode"""
    amap_api_key = os.environ["AMAP_API_KEY"]

    # 获取行政区域编码
    url = f"https://restapi.amap.com/v3/config/district?key={amap_api_key}&keywords={city}"
    reponse_json = request_with_retry(url, 3)
    if reponse_json['status'] == '1': 
        return reponse_json.get('districts')[0]['adcode']
    else:
        return None

def get_future_weather_use_acode(acode, date):
    amap_api_key = os.environ["AMAP_API_KEY"]
    url = f"https://restapi.amap.com/v3/weather/weatherInfo?key={amap_api_key}&city={acode}&extensions=all"
    response_json = request_with_retry(url)
    if response_json['status'] != '1':
        return "未查到天气情况"
    else:
        weather_json = response_json.get("forecasts")[0]
        weather_json["casts"] = [cast for cast in weather_json["casts"] if cast["date"] == date]
        if weather_json["casts"] == []:
            weather_json["casts"] = "未查到天气情况"
        return json.dumps(weather_json, ensure_ascii=False, indent=4)


def get_future_weather(city, date): 
    url = ""
    """获取城市天气"""
    amap_api_key = os.environ["AMAP_API_KEY"]

    acode = get_adcode(city)
    if acode == None:
        return "未查到天气情况"
    else:
       return get_future_weather_use_acode(acode, date)

In [14]:
# 工具3: get_foods
def get_foods_for_page(location, page, radius=10000, rating=4.5):
    """获取POI附近美食"""
    amap_api_key = os.environ["AMAP_API_KEY"]
    url = f"https://restapi.amap.com/v3/place/around?"
    params = {
        'key': amap_api_key,
        'location': location, 
        'types': "中餐厅",
        'radius': radius,
        'sorted': "weight",
        'offset': 10,
        'page': page
    }
    data = request_with_retry(url, params)
    if data == None:
        return "未查到相关餐厅"
    else:
        pois = data['pois']

        # 取评分超过4.5的
        good_foods = [{
            'name': poi['name'],
            'type': poi['type'],
            'address': poi['address'],
            'cityname': poi['cityname'],
            'tel': poi['tel'],
            'location': poi['location'],
            'distance': poi['distance'],
            'rating': poi['biz_ext']['rating'],
            'cost': poi['biz_ext']['cost']
        } for poi in pois if poi['biz_ext'].get('rating') and float(poi['biz_ext']['rating']) >= rating]
        return good_foods

def get_foods(location):
    all_foods = []
    for page in range(0, 4): 
        all_foods.extend(get_foods_for_page(location, page))
    return json.dumps(all_foods, ensure_ascii=False, indent=4)
        

In [15]:
# 工具测试
print(get_famous_spots("云南"))
print(get_foods('100.164000,25.694836'))
print(get_future_weather("大理白族自治州", "2026-02-06"))

[
    {
        "name": "大理古城",
        "address": "护国路172号",
        "cityname": "大理白族自治州",
        "tel": "4008721699",
        "location": "100.164000,25.694836",
        "rating": "4.8",
        "cost": []
    },
    {
        "name": "丽江古城",
        "address": "民主路与福慧路交叉口东南",
        "cityname": "丽江市",
        "tel": "0888-5111118",
        "location": "100.235517,26.870507",
        "rating": "4.9",
        "cost": []
    },
    {
        "name": "玉龙雪山国家级风景名胜区",
        "address": "旅游环线",
        "cityname": "丽江市",
        "tel": "0888-5131068",
        "location": "100.258558,27.098052",
        "rating": "4.8",
        "cost": []
    },
    {
        "name": "告庄西双景",
        "address": "曼泐路12号靠近达兰商业广场",
        "cityname": "西双版纳傣族自治州",
        "tel": "0691-2221600;0691-2221171",
        "location": "100.821392,22.007019",
        "rating": "4.7",
        "cost": []
    },
    {
        "name": "双廊古镇",
        "address": "S314(环海东路)",
        "cityname": "大理白族自治州",
        "tel"

# Schema定义

In [16]:
get_famous_spots_schema = {
    "type": "function",
    "function": {
        "name": "get_famous_spots",
        "description": "Get the famous spots of a sepcial region",
        "parameters": {
            "type": "object",
            "required": ["city"],
            "properties": {
                "city": {
                    "type": "string",
                    "description": "A city name like 云南"
                },
            }
        }
    }
}

In [22]:
get_foods_schema = {
    "type": "function",
    "function": {
        "name": "get_foods",
        "description": "Get the foods of a sepcial location",
        "parameters": {
            "type": "object",
            "required": ["location"],
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Location is longitude and latitude, like '100.164000,25.694836'."
                },
            }
        }
    }
}

In [18]:
get_future_weather_schema = {
    "type": "function",
    "function": {
        "name": "get_future_weather",
        "description": "Get the weather of a city for a particular day",
        "parameters": {
            "type": "object",
            "required": ["city", "date"],
            "properties": {
                "city": {
                    "type": "string",
                    "description": "A city name like 北京 or 上海"
                },
                "date": {
                    "type": "string",
                    "description": "The target future date, formatted as “yyyy-MM-dd, like 2026-02-02"
                }
            }
        }
    }
}

In [21]:
tools = [
    get_famous_spots_schema, 
    get_future_weather_schema,
    get_foods_schema
]

fn_map = {
    "get_famous_spots": get_famous_spots,
    "get_future_weather": get_future_weather,
    "get_foods": get_foods
}

# 工具注册和执行

In [94]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"]
    )


In [ ]:
messages = [] 
messages.append({
    "role": "system",
    "content": f"""
        你是一个专业的旅行规划师，请根据用户输入的地点和时间制定一份合理的旅行计划，包括每天行程、美食、天气等信息，你可以这样做：
        - 第一步，规划每天游玩的城市和景点
        - 第二步，提供游玩景点所在城市的天气
        - 第三步，推荐3个游玩景点附近美食
    """
})
messages.append ({
    "role": "user",
    "content": "请帮我规划一个2026年2月5日至2月7日的云南旅游，需要具体到城市、天、每天天气"
})

finish_reason = None
round_cnt = 0
while finish_reason is None or finish_reason == "tool_calls":
    round_cnt += 1
    resp = client.chat.completions.create(
        model="Qwen/Qwen3-235B-A22B-Instruct-2507",
        messages=messages, 
        tools=tools, 
        tool_choice="auto"
    )
    choice = resp.choices[0]
    finish_reason = choice.finish_reason
    if finish_reason == "tool_calls":
        messages.append({
            "role": "assistant",
            "tool_calls": [tool_call.model_dump() for tool_call in resp.choices[0].message.tool_calls]
        })
        print(f"Round {round_cnt}: {choice.message.tool_calls}")
        for tool_call in choice.message.tool_calls:
            print(tool_call.model_dump())
            tool_call_name = tool_call.function.name
            tool_call_arguments = tool_call.function.arguments
            tool_call_result = fn_map[tool_call_name](**json.loads(tool_call_arguments))
            print(tool_call_result)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_call.function.name,
                    "content": tool_call_result
                }
            )
    

In [98]:
# 打印结果
print(resp.choices[0].message.content)


### 云南3日旅行计划（2026年2月5日–2月7日）  
为您精心规划一趟融合古城风情、自然风光与地道美食的云南短途旅行，涵盖丽江、大理、昆明三地，每日天气晴朗，适宜出行。

---

#### **第一天：2026年2月5日（星期四）｜丽江古城 & 束河古镇**
**城市**：丽江市  
**天气**：晴，白天气温16°C，夜间1°C，西风1–3级，体感舒适，早晚温差大，注意保暖。  

**行程安排**：  
- **上午**：抵达丽江，前往【丽江古城】（世界文化遗产），漫步四方街，感受纳西族建筑与文化风情。  
- **中午**：在古城内享用午餐，推荐尝试当地特色腊排骨火锅。  
- **下午**：前往【束河古镇】，比丽江古城更安静古朴，适合拍照、品茶、体验慢生活。  
- **晚上**：返回丽江古城，欣赏夜景，感受灯火阑珊的古城魅力。

**推荐美食（丽江古城附近）**：  
1. **阿婆情腊排骨火锅（七一街店）**  
   - 特色：地道纳西风味，腊香浓郁  
   - 人均：¥65｜评分：4.5  

2. **樱花餐厅**  
   - 特色：融合云南菜，环境优雅  
   - 人均：¥87｜评分：4.7  

3. **云雪丽·木府土司文化 现炒云南菜**  
   - 特色：现点现炒，味道正宗  
   - 人均：¥84｜评分：4.7  

---

#### **第二天：2026年2月6日（星期五）｜大理古城 & 洱海**
**城市**：大理白族自治州（大理市）  
**天气**：晴，白天气温20°C，夜间2°C，西南风1–3级，阳光明媚，适合户外活动。

**行程安排**：  
- **上午**：乘车前往大理（约2小时车程），抵达后游览【大理古城】，漫步护国路，感受白族风情。  
- **中午**：在古城内用餐，推荐尝试创新云南菜或野生菌料理。  
- **下午**：环游【洱海】，可骑行或乘坐观光车，途经【双廊古镇】，欣赏苍山洱海美景。  
- **晚上**：入住洱海附近民宿，享受宁静夜晚。

**推荐美食（大理古城附近）**：  
1. **梁厨在大理·创新菜**  
   - 特色：融合创意菜，口味新颖  
   - 人均：¥61｜评分：4.5  

2. **尽善（百年古院餐厅）**  
   - 特色：百年老宅用餐，环境独特  